
# Lab 2 — ROS Graph Mastery: Nodes, Topics, Services, Parameters (with TurtleSim)

## What you’ll build
You will bring up **turtlesim** and then *prove* you understand the ROS computation graph by:
- listing and inspecting **nodes**
- introspecting **topics** (types, publishers/subscribers, live data)
- calling **services** and reading service types/arguments
- reading + setting **parameters** and validating system behavior

This lab is intentionally CLI-heavy: the goal is to make you comfortable diagnosing ROS systems from the terminal.

> **Important:** Most cells contain terminal commands. Run them in an Ubuntu terminal (VM is fine).

---

## Learning outcomes
By the end of this lab, you can:
1. Identify nodes and their ROS interfaces (pub/sub/services)
2. Trace data flow through topics, including message types and fields
3. Use services for request/response actions (spawn, clear, teleport, etc.)
4. Use parameters for runtime configuration and validate the effect

---

## Deliverables
Submit:
- A short PDF (or markdown) with screenshots + answers (see checklist at the end)
- Terminal outputs for selected commands (copy/paste is OK)



---
## Part 0 — Concept refresher

**Node**  
A node is a running process in ROS (like a program). Engineers design nodes to do one job well.

**Topic**  
A named channel for streaming messages (publish/subscribe). Good for continuous sensor data and continuous commands.

**Message**  
A data structure transmitted on a topic (e.g., `geometry_msgs/Twist`). Publisher and subscriber must agree on the message type.

**Service**  
A request/response interaction (call-and-reply). Use services for discrete operations: reset, spawn, clear, teleport, etc.

**Parameter**  
A configuration value stored on the Parameter Server (ROS1) and used by nodes to tune behavior (e.g., background color).

> **Engineering mindset:** Don’t just “run things.” Always verify: what nodes exist, what topics exist, what types, who publishes, who subscribes, what changed after your action.



---
## Part A — Nodes: bring up the system and identify processes

### A1) Start the ROS master and turtlesim
Open **Terminal 1**:


In [ ]:

# Terminal 1
roscore



Open **Terminal 2**:


In [ ]:

# Terminal 2
rosrun turtlesim turtlesim_node



### A2) List running nodes
Open **Terminal 3**:


In [ ]:

# Terminal 3
rosnode list



✅ **Checkpoint A:** You should see at least:
- `/rosout`
- `/turtlesim`



### A3) Add a teleop node and re-check
Open **Terminal 4**:


In [ ]:

# Terminal 4
rosrun turtlesim turtle_teleop_key



Return to Terminal 3 and run:


In [ ]:

rosnode list



✅ **Checkpoint B:** You should now see a teleop node too (commonly `/teleop_turtle`).



### A4) Remap a node name (practice)
Launch an additional turtlesim node but rename it using remapping.

Open **Terminal 5**:


In [ ]:

# Terminal 5
rosrun turtlesim turtlesim_node __name:=my_turtle



Now re-run:


In [ ]:

# Terminal 3
rosnode list



### A5) Inspect node interfaces
Use `rosnode info` to see what a node publishes/subscribes/provides as services.

Inspect your renamed node:


In [ ]:

# Terminal 3
rosnode info /my_turtle



**Mini-task A:** Also inspect the teleop node and write **two differences** you observe between `/teleop_turtle` and `/turtlesim` or `/my_turtle`.



---
## Part B — Topics: observe data flow and message types

### B0) Clean setup
Close the `my_turtle` terminal (Terminal 5) so you’re back to one turtlesim + teleop.

Keep these running:
- `roscore`
- `turtlesim_node`
- `turtle_teleop_key`



### B1) Visualize the graph with rqt_graph
Open **Terminal 5**:


In [ ]:

# Terminal 5
rqt_graph



✅ **Checkpoint C:** You should see teleop publishing to a velocity topic and turtlesim subscribing.

> Tip: If the graph is empty, hit refresh. If it looks “too small,” adjust the hide options and refresh.



### B2) List topics and get details
Open **Terminal 6**:


In [ ]:

# Terminal 6
rostopic list
rostopic list -v



### B3) Watch live messages on a topic
In Terminal 6, run:


In [ ]:

# Terminal 6
rostopic echo /turtle1/cmd_vel



Now move the turtle using the teleop terminal and observe the printed `Twist` messages.

**Mini-task B:** What changes in the message when you press:
- Up arrow (forward)
- Left arrow (rotate)



### B4) Identify the topic type and inspect message structure
In Terminal 6:


In [ ]:

# Terminal 6
rostopic type /turtle1/cmd_vel
rosmsg show geometry_msgs/Twist



✅ **Checkpoint D:** You should recognize that `Twist` contains two vectors: `linear` and `angular`, each with x/y/z fields.

### B5) Publish to a topic from the CLI (engineering injection test)
In a new terminal (Terminal 7), publish one message and exit:


In [ ]:

# Terminal 7 (publish once)
rostopic pub -1 /turtle1/cmd_vel geometry_msgs/Twist -- "[2.0, 0.0, 0.0]" "[0.0, 0.0, 1.8]"



Now publish continuously at 1 Hz:


In [ ]:

# Terminal 7 (publish at 1 Hz)
rostopic pub /turtle1/cmd_vel geometry_msgs/Twist -r 1 -- "[2.0, 0.0, 0.0]" "[0.0, 0.0, -1.8]"



✅ **Checkpoint E:** The turtle should move continuously (likely a circle/arc depending on values).

**Mini-task C:** Open `rqt_graph` and identify the new node created by `rostopic pub`. Who are the subscribers?



### B6) Observe pose updates
Open **Terminal 8**:


In [ ]:

# Terminal 8
rostopic echo /turtle1/pose



**Mini-task D:** While the turtle is moving, identify which fields in `/turtle1/pose` are changing continuously.



---
## Part C — Services: request/response operations

### C0) Keep only what you need
Keep running:
- `roscore`
- `turtlesim_node`
- `turtle_teleop_key` (optional)

Stop extra terminals like `rostopic pub` and `echo` if they clutter your view.



### C1) List available services
Open **Terminal 6** (or any new one):


In [ ]:

rosservice list



### C2) Inspect service types
Check the type for `/clear`:


In [ ]:

rosservice type /clear
rossrv show std_srvs/Empty



Now inspect `/spawn` (type + request/response fields):


In [ ]:

rosservice type /spawn
rosservice type /spawn | rossrv show



### C3) Call services from the CLI
Clear the drawing:


In [ ]:

rosservice call /clear



Spawn a second turtle (pick any pose you like):


In [ ]:

# Example spawn
rosservice call /spawn 2 2 0.2 ""



✅ **Checkpoint F:** You should see a new turtle appear (commonly `turtle2`).

**Mini-task E (engineering):** Use `rosservice list` again. What new services appeared after spawning `turtle2`?



---
## Part D — Parameters: runtime configuration and validation

### D1) List parameters


In [ ]:

rosparam list



### D2) Read the turtlesim background parameters


In [ ]:

rosparam get /turtlesim/background_r
rosparam get /turtlesim/background_g
rosparam get /turtlesim/background_b



### D3) Set a new background color (and force refresh)
In turtlesim, background color changes usually take effect after calling `/clear`.
Try setting a new red value:


In [ ]:

rosparam set /turtlesim/background_r 150
rosservice call /clear



✅ **Checkpoint G:** Background color should change.

### Mini-task F (choose one)
- Make the background **orange** (high red, medium green, low blue), OR
- Make the background **cyan** (low red, high green, high blue)

Record your chosen RGB values.



---
## Lab Exercise — Two-turtle motion challenge

Goal: demonstrate graph understanding by commanding two turtles **simultaneously** with different behaviors.

### Requirements
1. Set turtlesim background to **orange or cyan** (your choice).
2. Ensure `turtle1` and `turtle2` both exist (spawn turtle2 if needed).
3. Command:
   - `turtle1` to move in a **large clockwise circle continuously**
   - `turtle2` to move in a **smaller counter-clockwise circle for exactly one full loop**, then stop

### Constraints (engineering-friendly)
- You must use **CLI tools** for motion injection:
  - `rostopic pub` (rate-controlled) for the continuous motion
  - either `rostopic pub -1` (one-shot) + timing, OR a brief scripted loop, for the one-loop motion
- You must prove correctness using:
  - `rqt_graph` screenshot showing both control flows, and
  - at least one `rostopic echo` showing messages going to both turtles

### Hints
- Clockwise vs counter-clockwise is typically controlled by the sign of `angular.z`.
- Large vs small circle can be controlled by the ratio between `linear.x` and `angular.z` (smaller radius usually means stronger angular relative to linear).
- You can remap a `rostopic pub` to target `/turtle2/cmd_vel`.



---
## Submission checklist

Include:
1. Screenshot: turtlesim showing your chosen background color.
2. Screenshot: `rqt_graph` showing **two** command publishers and **two** turtle cmd_vel topics.
3. Terminal output:
   - `rosnode list`
   - `rostopic list -v`
   - `rosservice list`
   - `rosparam list`
4. Short answers (2–4 sentences each):
   - What is the difference between **topics** and **services**?
   - Why does changing background parameters require calling `/clear` to visually update?
   - What is one debugging step you would do first if the turtle does not move?



---
## Appendix — Ubuntu 24.04 Track (ROS 2 Jazzy equivalents)

If you are on **Ubuntu 24.04**, use **ROS 2 Jazzy** and the equivalent commands below.

### Bring up turtlesim
```bash
ros2 run turtlesim turtlesim_node
ros2 run turtlesim turtle_teleop_key
```

### Nodes / Topics
```bash
ros2 node list
ros2 node info /turtlesim
ros2 topic list
ros2 topic info /turtle1/cmd_vel
ros2 topic echo --once /turtle1/pose
```

### Publish Twist (ROS 2 uses YAML)
```bash
ros2 topic pub --once /turtle1/cmd_vel geometry_msgs/msg/Twist "{linear: {x: 2.0, y: 0.0, z: 0.0}, angular: {z: 1.8}}"
ros2 topic pub /turtle1/cmd_vel geometry_msgs/msg/Twist "{linear: {x: 2.0}, angular: {z: -1.8}}" -r 1
```

### Services (spawn, set_pen, clear)
```bash
ros2 service list
ros2 service type /spawn
ros2 interface show turtlesim/srv/Spawn

ros2 service call /spawn turtlesim/srv/Spawn "{x: 2.0, y: 2.0, theta: 0.2, name: 'turtle2'}"
ros2 service call /clear std_srvs/srv/Empty "{}"
ros2 service call /turtle1/set_pen turtlesim/srv/SetPen "{r: 0, g: 255, b: 255, width: 5, off: 0}"
```

### Parameters (ROS 2 uses `ros2 param`)
```bash
ros2 param list
ros2 param get /turtlesim background_r
ros2 param set /turtlesim background_r 150
ros2 service call /clear std_srvs/srv/Empty "{}"
```
